# JAX — NumPy That Runs at C Speed and Knows Calculus

---

## What Is JAX?

JAX is an open-source numerical computing library from Google Research, released in 2018. Think of it as **NumPy with three superpowers**:

1. **`jax.grad`** — automatic differentiation: compute derivatives of any function automatically
2. **`jax.jit`** — JIT (Just-In-Time) compilation: compile Python functions to XLA machine code (2-100x faster)
3. **`jax.vmap`** — automatic vectorization: apply a function to every element in a batch without writing loops
4. **`jax.pmap`** — parallel execution: automatically run across multiple GPUs/TPUs

### Real-World Analogy

Regular NumPy is like a **very organized spreadsheet** — great for math, but slow for repeated heavy computation. JAX is like that spreadsheet plugged into a **supercomputer with a calculus engine**: it can compute derivatives automatically (no hand math), run your code as compiled machine instructions (JIT), and process thousands of examples simultaneously (vmap). It's what powers Google DeepMind's research at scale.

---

## Why Learn JAX?

| Use Case | Why JAX Wins |
|---|---|
| ML research | `grad` + `jit` = fast, clean research code |
| Custom optimizers | Differentiate through any Python function |
| Physics simulations | JIT-compiled ODEs, PDEs run as fast as Fortran |
| Bayesian inference | Probabilistic libraries (NumPyro, BlackJAX) built on JAX |
| Transformer models | DeepMind, Google Brain use JAX+Flax for models like Gemini |
| TPU training | JAX has first-class TPU support (TensorFlow does too; PyTorch lags) |

---

## Prerequisites

- NumPy (JAX's API mirrors NumPy almost exactly)
- Basic calculus — knowing what a gradient/derivative is
- Basic neural network concepts (loss function, gradient descent)

---

## Table of Contents

1. Installation & Setup
2. JAX Arrays — Like NumPy but Immutable
3. `jax.grad` — Automatic Differentiation
4. `jax.jit` — JIT Compilation for Speed
5. `jax.vmap` — Automatic Vectorization
6. `jax.pmap` — Multi-Device Parallelism (Overview)
7. Random Numbers — JAX's Functional PRNG
8. The Pure Function Constraint
9. Building a Neural Network from Scratch
10. Higher-Level Libraries: Flax & Optax
11. Mini Project — Training a Neural Net with JAX
12. Common Pitfalls
13. Interview Q&A
14. Resources
15. Summary & What's Next

---

**Official Docs:** https://docs.jax.dev/  
**GitHub:** https://github.com/google/jax  
**Paper:** https://arxiv.org/abs/2012.16867  
**YouTube — JAX crash course:** https://www.youtube.com/watch?v=iDxJxIyzSiM  
**Flax (NN library on JAX):** https://flax.readthedocs.io/  
**Optax (optimizers for JAX):** https://optax.readthedocs.io/  

## 1. Installation & Setup

```bash
# CPU-only (works everywhere)
pip install jax jaxlib

# CUDA 12 GPU support
pip install -U jax[cuda12]

# TPU support
pip install -U jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
```

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, random
from functools import partial
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print(f"JAX version:  {jax.__version__}")
print(f"Devices:      {jax.devices()}")
print(f"Default device: {jax.default_backend()}")

## 2. JAX Arrays — Like NumPy but Immutable

`jax.numpy` (`jnp`) mirrors NumPy almost exactly. The key differences:

| Feature | NumPy | JAX |
|---|---|---|
| Mutability | Mutable (`a[0] = 5`) | **Immutable** (`a.at[0].set(5)` → new array) |
| Device | CPU only | CPU, GPU, or TPU |
| Lazy evaluation | Eager | Can be JIT-compiled (lazy) |
| Random numbers | Stateful (`np.random.seed()`) | **Functional** (explicit keys) |
| Out-of-place ops | Some in-place ops | All ops return new arrays |

In [ ]:
# JAX arrays work just like NumPy
a = jnp.array([1.0, 2.0, 3.0, 4.0])
b = jnp.array([5.0, 6.0, 7.0, 8.0])

print("Element-wise ops:")
print("  a + b:", a + b)
print("  a * b:", a * b)
print("  jnp.sin(a):", jnp.sin(a))
print("  jnp.sum(a):", jnp.sum(a))

# Matrices
M = jnp.arange(9.0).reshape(3, 3)
print("\nMatrix:\n", M)
print("Matmul:\n", M @ M)
print("Transpose:\n", M.T)

# JAX arrays are IMMUTABLE — can't do a[0] = 5
# Instead use .at[].set() which returns a NEW array
a_modified = a.at[0].set(99.0)
print("\nOriginal a:", a)           # unchanged!
print("Modified:   ", a_modified)   # new array with 99 at index 0

# Other .at[] operations
print("Add at index 1:", a.at[1].add(10.0))   # [1, 12, 3, 4]
print("Multiply at [2]:",a.at[2].mul(100.0))  # [1, 2, 300, 4]

In [ ]:
# JAX vs NumPy performance on the same computation

n = 5000
A_np = np.random.randn(n, n).astype(np.float32)
A_jx = jnp.array(A_np)

# NumPy
t0 = time.time()
for _ in range(3):
    _ = np.dot(A_np, A_np)
np_time = time.time() - t0

# JAX (first call compiles, subsequent calls are fast)
_ = jnp.dot(A_jx, A_jx).block_until_ready()  # warm up
t0 = time.time()
for _ in range(3):
    _ = jnp.dot(A_jx, A_jx).block_until_ready()
jx_time = time.time() - t0

print(f"Matrix multiply ({n}×{n}), 3 repeats:")
print(f"  NumPy: {np_time:.3f}s")
print(f"  JAX:   {jx_time:.3f}s")
print(f"  Speedup: {np_time/jx_time:.1f}x")
print("(JAX speedup is much larger on GPU/TPU)")

## 3. `jax.grad` — Automatic Differentiation

`jax.grad(f)` returns a **new function** that computes the gradient of `f` with respect to its first argument.

**Rule:** The function `f` must return a **scalar** (single number). The gradient is a vector/tensor with the same shape as the input, where each element is `∂f/∂x_i`.

This is the foundation of all neural network training — computing `∂Loss/∂weights` for every weight.

In [ ]:
# ---- Basic gradient computation ----

# f(x) = x^2 → f'(x) = 2x
def square(x):
    return x ** 2

dsquare = grad(square)   # dsquare is a function that computes df/dx
print(f"f(3.0) = {square(3.0)}")                     # 9.0
print(f"f'(3.0) = {dsquare(3.0)}")                   # 6.0  (2×3)
print(f"f''(3.0) = {grad(grad(square))(3.0)}")       # 2.0  (d²f/dx²)
print(f"f'''(3.0) = {grad(grad(grad(square)))(3.0)}") # 0.0

# ---- Gradient of a multivariate function ----
# f(x, y) = x^2 + 3*x*y + y^3
# ∂f/∂x = 2x + 3y
# ∂f/∂y = 3x + 3y^2

def f_multi(params):   # params = [x, y]
    x, y = params[0], params[1]
    return x**2 + 3*x*y + y**3

grad_f = grad(f_multi)  # gradient w.r.t. the whole params vector
params = jnp.array([2.0, 1.0])  # x=2, y=1
print(f"\nf(2, 1) = {f_multi(params)}")    # 4 + 6 + 1 = 11
print(f"∂f/∂x at (2,1) = {grad_f(params)[0]}")  # 2*2 + 3*1 = 7
print(f"∂f/∂y at (2,1) = {grad_f(params)[1]}")  # 3*2 + 3*1^2 = 9

In [ ]:
# ---- Value AND gradient together ----
# jax.value_and_grad returns (f(x), grad_f(x)) in one pass — more efficient

value_and_grad_f = jax.value_and_grad(f_multi)
val, grads = value_and_grad_f(params)
print(f"Value: {val}, Gradient: {grads}")

# ---- Gradient w.r.t. a specific argument using argnums ----
def loss(weights, X, y):
    """Linear regression MSE loss."""
    pred = X @ weights
    return jnp.mean((pred - y) ** 2)

# grad w.r.t. first argument (weights) by default (argnums=0)
grad_weights = grad(loss, argnums=0)

key = random.PRNGKey(42)
w_true = jnp.array([2.0, -1.0, 0.5])
X_demo = random.normal(key, (100, 3))
y_demo = X_demo @ w_true + 0.1 * random.normal(key, (100,))
w_init = jnp.zeros(3)

g = grad_weights(w_init, X_demo, y_demo)
print(f"\nGradient of MSE loss w.r.t. weights: {g}")
print("(This is what each SGD step uses to update weights)")

## 4. `jax.jit` — JIT Compilation for Speed

`jax.jit(f)` compiles a function into **XLA (Accelerated Linear Algebra)** machine code. The first call traces and compiles (slow), all subsequent calls use the compiled version (fast).

**XLA** fuses operations together, eliminating Python overhead and intermediate memory allocations. On GPU/TPU, it can be 10-100x faster than eager execution.

In [ ]:
# ==================================================
# JIT benchmark
# ==================================================

def relu(x):
    return jnp.maximum(0, x)

def dense_forward(W, b, x):
    return relu(x @ W + b)

# JIT-compiled version
dense_jit = jit(dense_forward)

key = random.PRNGKey(0)
W = random.normal(key, (1000, 512))
b = jnp.zeros(512)
x = random.normal(key, (256, 1000))  # batch of 256, 1000 features

# Warm up JIT compilation
_ = dense_jit(W, b, x).block_until_ready()

# Benchmark
N = 100
t0 = time.time()
for _ in range(N):
    dense_forward(W, b, x).block_until_ready()
eager_t = time.time() - t0

t0 = time.time()
for _ in range(N):
    dense_jit(W, b, x).block_until_ready()
jit_t = time.time() - t0

print(f"Forward pass × {N}:")
print(f"  Eager (no JIT): {eager_t:.3f}s")
print(f"  JIT compiled:   {jit_t:.3f}s")
print(f"  Speedup: {eager_t/jit_t:.1f}x")

# You can also use it as a decorator
@jit
def sigmoid(x):
    return 1.0 / (1.0 + jnp.exp(-x))

print("\nsigmoid([0, 1, -1]):", sigmoid(jnp.array([0.0, 1.0, -1.0])))

In [ ]:
# JIT + grad = fast gradient computation
# This is exactly how neural network training works in JAX!

def mse_loss(weights, X, y):
    pred = X @ weights
    return jnp.mean((pred - y) ** 2)

# Combine grad + jit: compute gradient AND compile it
# jit(grad(...)) is the standard JAX pattern for fast training steps
grad_fn = jit(grad(mse_loss))

# Also useful: jit(value_and_grad(...))
val_grad_fn = jit(jax.value_and_grad(mse_loss))

# Quick training loop
key = random.PRNGKey(42)
w_true = jnp.array([3.0, -2.0, 1.0])
X_d = random.normal(key, (500, 3))
y_d = X_d @ w_true + 0.1 * random.normal(key, (500,))

weights = jnp.zeros(3)  # start from all zeros
lr = 0.1

print("Learning y = 3x0 - 2x1 + 1x2  ...")
for step in range(50):
    loss_val, grads = val_grad_fn(weights, X_d, y_d)
    weights = weights - lr * grads  # gradient descent update
    if (step + 1) % 10 == 0:
        print(f"Step {step+1:2d}: loss={loss_val:.5f}  w={weights}")

print(f"\nTrue weights:     {w_true}")
print(f"Recovered weights: {weights.round(3)}")

## 5. `jax.vmap` — Automatic Vectorization

**The problem:** You write a function that processes ONE sample. To process a batch, you'd normally have to:
- Add batch dimensions manually
- Rewrite indexing logic
- Use explicit loops (slow)

**JAX's solution:** `vmap(f)` automatically maps `f` over a batch dimension. You write code for one sample; `vmap` handles the batch. This is like numpy broadcasting but for arbitrary functions.

In [ ]:
# ---- Example 1: vectorize a scalar function ----

def compute_for_one(x, y):
    """Process a SINGLE sample: compute x^2 + sin(y)."""
    return x**2 + jnp.sin(y)

# To apply to a batch, we'd normally need a loop:
X_batch = jnp.array([1.0, 2.0, 3.0, 4.0])
Y_batch = jnp.array([0.0, 0.5, 1.0, 1.5])

# Old way (Python loop):
result_loop = jnp.array([compute_for_one(x, y) for x, y in zip(X_batch, Y_batch)])

# JAX way (vmap — vectorized automatically):
compute_batch = vmap(compute_for_one)   # maps over first axis of each argument
result_vmap = compute_batch(X_batch, Y_batch)

print("Results match:", jnp.allclose(result_loop, result_vmap))
print("Results:", result_vmap)

# ---- Example 2: vectorize a neural network forward pass ----

def single_forward(W1, b1, W2, b2, x):
    """Forward pass for ONE sample x (shape: [D_in])."""
    h = jnp.tanh(W1 @ x + b1)   # hidden layer
    return W2 @ h + b2            # output

# Vectorize over the LAST argument (x), keep W1, b1, W2, b2 fixed
# in_axes=(None, None, None, None, 0) means:
#   first 4 args are NOT batched (None), last arg is batched over axis 0
batch_forward = vmap(single_forward, in_axes=(None, None, None, None, 0))

key = random.PRNGKey(0)
D_in, D_hidden, D_out = 10, 32, 5
W1 = random.normal(key, (D_hidden, D_in))
b1 = jnp.zeros(D_hidden)
W2 = random.normal(key, (D_out, D_hidden))
b2 = jnp.zeros(D_out)

X_batch2 = random.normal(key, (64, D_in))   # 64 samples

output = batch_forward(W1, b1, W2, b2, X_batch2)
print(f"\nBatch forward:  input {X_batch2.shape} → output {output.shape}")

## 6. `jax.pmap` — Multi-Device Parallelism (Overview)

`pmap` is like `vmap` but distributes computation across **multiple devices** (multiple GPUs or TPUs).

```python
# Run on 8 GPUs: split batch across GPUs, compute in parallel
parallel_fn = jax.pmap(train_step)

# Replicate params to each device
params_replicated = jax.device_put_replicated(params, jax.devices())

# Split batch: (8×32, ...) → 8 devices × 32 samples each
batch_per_device = batch.reshape(8, 32, *batch.shape[1:])

# Run in parallel — all-reduces gradients automatically
params_replicated, losses = parallel_fn(params_replicated, batch_per_device)
```

This is why JAX is the framework of choice for training large models on TPU pods (thousands of devices). `pmap` handles the **communication** (all-reduce for gradients) automatically.

## 7. Random Numbers — JAX's Functional PRNG

NumPy uses **stateful** random number generation: a hidden global state is updated each time you call `np.random.randn()`. This makes code hard to reproduce and doesn't work well with JIT.

JAX uses **functional** PRNG: you explicitly pass and split **keys**. The same key always produces the same result — perfect for reproducibility and JIT compatibility.

In [ ]:
# JAX random numbers are functional — same key = same output
key = random.PRNGKey(42)  # create a key from a seed

# Using the same key twice gives IDENTICAL results
a = random.normal(key, (3,))
b = random.normal(key, (3,))
print("Same key, same result:", jnp.allclose(a, b))  # True!

# To get DIFFERENT numbers, split the key
key, subkey1, subkey2 = random.split(key, num=3)  # split into 3 independent keys

c = random.normal(subkey1, (3,))
d = random.normal(subkey2, (3,))
print("Different keys, different results:", jnp.allclose(c, d))  # False
print(f"c = {c}")
print(f"d = {d}")

# Standard pattern in JAX code:
def init_model(key, input_dim, hidden_dim, output_dim):
    """Initialize model parameters using explicit keys."""
    key_w1, key_b1, key_w2, key_b2 = random.split(key, 4)
    return {
        'W1': random.normal(key_w1, (hidden_dim, input_dim)) * 0.01,
        'b1': jnp.zeros(hidden_dim),
        'W2': random.normal(key_w2, (output_dim, hidden_dim)) * 0.01,
        'b2': jnp.zeros(output_dim)
    }

params = init_model(random.PRNGKey(0), 10, 64, 5)
print(f"\nModel params:")
for k, v in params.items():
    print(f"  {k}: shape={v.shape}")

## 8. The Pure Function Constraint

For `jit`, `grad`, and `vmap` to work correctly, functions must be **pure**: same input → same output, no side effects.

**What you CANNOT do inside a JIT-compiled function:**
- Mutate Python variables (use JAX arrays)
- Print intermediate tensor values (only traced values are printed during compilation)
- Use Python control flow based on JAX tensor VALUES (use `jax.lax.cond`, `jax.lax.scan`)
- Modify global state

**What you CAN do:**
- Any `jnp` operations
- Python control flow based on SHAPES (not values)
- `jax.debug.print()` for debugging inside JIT

In [ ]:
# Pure function: no side effects, same input → same output
@jit
def pure_fn(x):
    return jnp.sum(x ** 2)

print("Pure function:", pure_fn(jnp.array([1.0, 2.0, 3.0])))

# Debugging inside JIT with jax.debug.print (prints during EXECUTION, not tracing)
@jit
def debug_fn(x):
    jax.debug.print("Inside JIT: x={x}", x=x)
    return jnp.sum(x)

debug_fn(jnp.array([1.0, 2.0, 3.0]))

# Conditional inside JIT: use jax.lax.cond instead of Python if
@jit
def abs_val(x):
    # Python `if x >= 0` would fail in JIT because x is a traced value!
    # Use lax.cond(condition, true_fn, false_fn, operand)
    return jax.lax.cond(
        x >= 0,
        lambda x: x,     # if true: return x
        lambda x: -x,    # if false: return -x
        x
    )

print("abs_val(-3.0):", abs_val(-3.0))
print("abs_val(5.0):",  abs_val(5.0))

# Loop inside JIT: use jax.lax.scan instead of Python for
# (lax.scan is like functools.reduce, very efficient)
@jit
def running_sum(xs):
    def accumulate(carry, x):
        return carry + x, carry + x  # (new_carry, output)
    final, cumsum = jax.lax.scan(accumulate, 0.0, xs)
    return cumsum

xs = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0])
print("Cumulative sum:", running_sum(xs))  # [1, 3, 6, 10, 15]

## 9. Building a Neural Network from Scratch in JAX

JAX has no built-in `nn.Module`. A network is just:
- **Parameters**: a dictionary (or nested list) of `jnp` arrays
- **Forward function**: a pure function `forward(params, x) → output`
- **Loss function**: a pure function `loss(params, X, y) → scalar`
- **Update function**: `new_params = params - lr * grad(loss)(params, X, y)`

In [ ]:
# ---- Define the network ----

def init_mlp(key, layer_sizes):
    """Initialize an MLP with the given layer sizes.
    Returns a list of (W, b) tuples.
    """
    params = []
    for in_size, out_size in zip(layer_sizes[:-1], layer_sizes[1:]):
        key, w_key = random.split(key)
        # He initialization: scale by sqrt(2/fan_in)
        W = random.normal(w_key, (in_size, out_size)) * jnp.sqrt(2.0 / in_size)
        b = jnp.zeros(out_size)
        params.append((W, b))
    return params


def mlp_forward(params, x):
    """Forward pass for one sample. Uses ReLU for hidden, linear for output."""
    activations = x
    for i, (W, b) in enumerate(params):
        preactivation = activations @ W + b
        if i < len(params) - 1:   # hidden layers
            activations = jax.nn.relu(preactivation)
        else:                      # output layer
            activations = preactivation
    return activations


# Vectorize over batch
batch_mlp = vmap(mlp_forward, in_axes=(None, 0))


def cross_entropy_loss(params, X, y):
    """Softmax cross-entropy loss for multi-class classification."""
    logits = batch_mlp(params, X)                    # (batch, n_classes)
    log_probs = jax.nn.log_softmax(logits, axis=-1)  # numerically stable
    return -jnp.mean(log_probs[jnp.arange(len(y)), y])


@jit
def train_step(params, X, y, lr=0.01):
    """One gradient descent step. Returns (updated_params, loss)."""
    loss, grads = jax.value_and_grad(cross_entropy_loss)(params, X, y)
    # Gradient descent: subtract lr * gradient for each weight
    new_params = [(W - lr * dW, b - lr * db)
                  for (W, b), (dW, db) in zip(params, grads)]
    return new_params, loss


def accuracy(params, X, y):
    """Compute classification accuracy."""
    logits = batch_mlp(params, X)
    preds  = jnp.argmax(logits, axis=-1)
    return jnp.mean(preds == y)


print("MLP from scratch defined. Architecture: 20 → 128 → 64 → 10")

In [ ]:
# ---- Train the MLP ----

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
X_mc, y_mc = make_classification(n_samples=2000, n_features=20, n_classes=10,
                                   n_informative=15, random_state=0)
X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(X_mc, y_mc, test_size=0.2)

scaler = StandardScaler()
X_mc_tr = scaler.fit_transform(X_mc_tr).astype(np.float32)
X_mc_te = scaler.transform(X_mc_te).astype(np.float32)
y_mc_tr = y_mc_tr.astype(np.int32)
y_mc_te = y_mc_te.astype(np.int32)

# Convert to JAX arrays
X_tr_j = jnp.array(X_mc_tr); y_tr_j = jnp.array(y_mc_tr)
X_te_j = jnp.array(X_mc_te); y_te_j = jnp.array(y_mc_te)

# Initialize model
key = random.PRNGKey(42)
params = init_mlp(key, layer_sizes=[20, 128, 64, 10])

# Training loop
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.01
N = len(X_mc_tr)

loss_history = []
print("Training MLP from scratch with JAX...")

for epoch in range(EPOCHS):
    # Shuffle data each epoch (manually, since JAX is functional)
    key, perm_key = random.split(key)
    perm = random.permutation(perm_key, N)
    X_shuffled = X_tr_j[perm]
    y_shuffled = y_tr_j[perm]

    epoch_loss = 0.0; n_batches = 0
    for i in range(0, N - BATCH_SIZE, BATCH_SIZE):
        Xb = X_shuffled[i:i+BATCH_SIZE]
        yb = y_shuffled[i:i+BATCH_SIZE]
        params, loss = train_step(params, Xb, yb, lr=LR)
        epoch_loss += loss; n_batches += 1

    loss_history.append(float(epoch_loss / n_batches))
    if (epoch + 1) % 20 == 0:
        tr_acc = accuracy(params, X_tr_j, y_tr_j)
        te_acc = accuracy(params, X_te_j, y_te_j)
        print(f"Epoch {epoch+1:3d} | Loss: {loss_history[-1]:.4f} | "
              f"Train Acc: {float(tr_acc):.3f} | Test Acc: {float(te_acc):.3f}")

# Plot
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='steelblue')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.title('JAX MLP Training Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Higher-Level Libraries: Flax & Optax

Writing neural networks from scratch in raw JAX (as above) gets tedious. The ecosystem provides two key libraries:

### Flax — Neural Network Library for JAX

Flax provides `nn.Module` — similar to PyTorch's module system but adapted to JAX's functional style. Parameters are stored **outside** the model (explicit state) rather than inside (as in PyTorch).

### Optax — Gradient Processing and Optimization

Optax provides all the optimizers you'd expect: `adam`, `sgd`, `adamw`, `rmsprop`, `lars`, `lamb`, plus gradient clipping, scheduling, and composition.

In [ ]:
try:
    import flax.linen as nn
    import optax
    from flax.training import train_state
    HAS_FLAX = True
except ImportError:
    HAS_FLAX = False
    print("Install: pip install flax optax")

if HAS_FLAX:
    # ==================================================
    # Flax model definition (like nn.Module in PyTorch)
    # ==================================================
    class FlaxMLP(nn.Module):
        hidden_size: int
        num_classes: int
        dropout_rate: float = 0.2

        @nn.compact
        def __call__(self, x, training: bool = False):
            x = nn.Dense(self.hidden_size)(x)
            x = nn.relu(x)
            x = nn.Dropout(self.dropout_rate, deterministic=not training)(x)
            x = nn.Dense(self.hidden_size // 2)(x)
            x = nn.relu(x)
            x = nn.Dense(self.num_classes)(x)
            return x

    # Initialize model
    model = FlaxMLP(hidden_size=128, num_classes=10)
    key = random.PRNGKey(0)
    dummy_input = jnp.ones((1, 20))  # to infer shapes

    # params are SEPARATE from the model — pure functional!
    variables = model.init(key, dummy_input)
    params_flax = variables['params']
    print("Flax model initialized!")
    print("Param shapes:", jax.tree_util.tree_map(lambda p: p.shape, params_flax))

    # Optax optimizer
    tx = optax.adam(learning_rate=0.001)
    opt_state = tx.init(params_flax)

    # Training state (convenience wrapper: bundles params + opt_state)
    state = train_state.TrainState.create(
        apply_fn=model.apply,
        params=params_flax,
        tx=tx
    )

    @jit
    def flax_train_step(state, X, y):
        def loss_fn(params):
            logits = state.apply_fn({'params': params}, X, training=True)
            loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
            return loss, logits

        (loss, logits), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        state = state.apply_gradients(grads=grads)  # update params + opt state
        acc = jnp.mean(jnp.argmax(logits, axis=-1) == y)
        return state, loss, acc

    # Run a few steps to verify
    for step in range(10):
        state, loss, acc = flax_train_step(state, X_tr_j[:64], y_tr_j[:64])

    print(f"\nAfter 10 steps: loss={float(loss):.4f}, acc={float(acc):.3f}")
    print("Flax + Optax integration working!")

## 11. Mini Project — Training a Full MLP with JAX + Optax

### The Task

Train a classifier on a multi-class dataset using:
- JAX arrays and functional programming
- Adam optimizer via Optax
- JIT-compiled training step
- Learning rate schedule (cosine decay)

In [ ]:
try:
    import optax
    HAS_OPTAX = True
except ImportError:
    HAS_OPTAX = False
    print("Install optax: pip install optax")

if HAS_OPTAX:
    # ==================================================
    # Dataset: breast cancer classification (2 classes)
    # ==================================================
    from sklearn.datasets import load_breast_cancer

    data = load_breast_cancer()
    X_bc = data.data.astype(np.float32)
    y_bc = data.target.astype(np.int32)

    X_bc_tr, X_bc_te, y_bc_tr, y_bc_te = train_test_split(
        X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
    )
    sc = StandardScaler()
    X_bc_tr = jnp.array(sc.fit_transform(X_bc_tr))
    X_bc_te = jnp.array(sc.transform(X_bc_te))
    y_bc_tr = jnp.array(y_bc_tr)
    y_bc_te = jnp.array(y_bc_te)

    print(f"Dataset: {X_bc_tr.shape} train, {X_bc_te.shape} test")
    print(f"Classes: {jnp.unique(y_bc_tr)}")

    # ==================================================
    # Model, optimizer, loss
    # ==================================================
    N_FEAT = X_bc_tr.shape[1]
    N_EPOCHS = 200
    BATCH = 32

    params_bc = init_mlp(random.PRNGKey(0), [N_FEAT, 64, 32, 2])

    # Cosine decay LR schedule
    n_steps = N_EPOCHS * (len(X_bc_tr) // BATCH)
    schedule = optax.cosine_decay_schedule(init_value=0.01, decay_steps=n_steps)
    optimizer = optax.adam(schedule)
    opt_state = optimizer.init(params_bc)

    def ce_loss_bc(params, X, y):
        logits = batch_mlp(params, X)
        return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

    @jit
    def step(params, opt_state, Xb, yb):
        loss, grads = jax.value_and_grad(ce_loss_bc)(params, Xb, yb)
        updates, new_opt_state = optimizer.update(grads, opt_state)
        new_params = optax.apply_updates(params, updates)
        return new_params, new_opt_state, loss

    # ==================================================
    # Training loop
    # ==================================================
    losses, accs_tr, accs_te = [], [], []
    key_bc = random.PRNGKey(1)

    for epoch in range(N_EPOCHS):
        key_bc, perm_key = random.split(key_bc)
        perm = random.permutation(perm_key, len(X_bc_tr))
        Xs, ys = X_bc_tr[perm], y_bc_tr[perm]

        epoch_loss = 0.0; n_b = 0
        for i in range(0, len(Xs) - BATCH, BATCH):
            params_bc, opt_state, loss = step(params_bc, opt_state, Xs[i:i+BATCH], ys[i:i+BATCH])
            epoch_loss += float(loss); n_b += 1

        losses.append(epoch_loss / n_b)
        accs_tr.append(float(accuracy(params_bc, X_bc_tr, y_bc_tr)))
        accs_te.append(float(accuracy(params_bc, X_bc_te, y_bc_te)))

        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {losses[-1]:.4f} | "
                  f"Train: {accs_tr[-1]:.3f} | Test: {accs_te[-1]:.3f}")

    # ==================================================
    # Plot
    # ==================================================
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(losses, color='steelblue')
    axes[0].set_title('Training Loss (Cosine LR Schedule)'); axes[0].set_xlabel('Epoch')
    axes[0].grid(alpha=0.3)

    axes[1].plot(accs_tr, label='Train', color='blue')
    axes[1].plot(accs_te, label='Test',  color='red')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('JAX + Optax: Breast Cancer Classifier', y=1.01)
    plt.tight_layout()
    plt.show()

    print(f"\nFinal Test Accuracy: {accs_te[-1]:.4f}")

## 12. Common Pitfalls

### Pitfall 1: Reusing the Same Key

```python
# WRONG — reusing the same key gives identical (correlated) results
key = random.PRNGKey(0)
W1 = random.normal(key, (10, 5))
W2 = random.normal(key, (5, 3))  # same values as W1's first 15 elements!

# CORRECT — always split
key, subkey1, subkey2 = random.split(key, 3)
W1 = random.normal(subkey1, (10, 5))
W2 = random.normal(subkey2, (5, 3))
```

### Pitfall 2: Forgetting `.block_until_ready()` in Benchmarks

JAX uses **asynchronous dispatch** — computations are launched but may not be complete yet. When timing, always call `.block_until_ready()` to wait for completion:
```python
result = jnp.dot(A, B).block_until_ready()  # waits for GPU to finish
```

### Pitfall 3: Python Side Effects in JIT

```python
counter = 0  # global mutable state

@jit
def bad_fn(x):
    global counter
    counter += 1  # only increments during TRACING, not every call!
    return x * 2

# After 5 calls: counter = 1 (not 5!)
```

### Pitfall 4: Using Python `if` on Traced Values

```python
@jit
def bad_conditional(x):
    if x > 0:  # ERROR! x is abstract during tracing, can't evaluate this
        return x
    return -x

@jit
def good_conditional(x):
    return jax.lax.cond(x > 0, lambda x: x, lambda x: -x, x)  # CORRECT
```

### Pitfall 5: In-Place Mutation

```python
@jit
def bad(x):
    x[0] = 5.0  # ERROR! JAX arrays are immutable
    return x

@jit
def good(x):
    return x.at[0].set(5.0)  # CORRECT — returns new array
```

## 13. Interview Q&A

---

**Q1: What are the four key transformations in JAX?**

> 1. **`jax.grad`** — automatic differentiation: returns a function that computes the gradient of another function
> 2. **`jax.jit`** — JIT compilation via XLA: compiles a Python function to fast machine code
> 3. **`jax.vmap`** — vectorization: automatically maps a single-sample function over a batch dimension
> 4. **`jax.pmap`** — parallelism: maps a function across multiple devices (GPUs/TPUs), with automatic all-reduce for gradients
> 
> These can be composed: `jit(vmap(grad(loss)))` gives you a JIT-compiled, vectorized gradient computation.

---

**Q2: What is XLA and why does JAX use it?**

> XLA (Accelerated Linear Algebra) is Google's domain-specific compiler for linear algebra computations. When JAX's `@jit` compiles a function, it traces the function to build an XLA computation graph, then XLA:
> 1. **Fuses operations** (e.g., `relu(matmul(x, W) + b)` becomes one kernel, not three)
> 2. **Eliminates unnecessary memory allocations** (intermediate results stay in registers)
> 3. **Optimizes for the target hardware** (CPU, GPU, or TPU)
> 
> Result: Python code runs as fast as hand-optimized C++/CUDA.

---

**Q3: Why does JAX use functional random number generation? What's wrong with NumPy's approach?**

> NumPy uses a **global hidden state** (modified by `np.random.seed()`). This breaks JIT compilation (hidden state is a side effect), makes it hard to reproduce results in multi-threaded code, and is hard to reason about. JAX's approach: explicit **PRNG keys** that you pass around explicitly. Same key → always same result. Different keys (via `random.split()`) → independent random numbers. This is **pure**: the function's output depends only on its arguments, enabling JIT and reproducibility.

---

**Q4: What does `vmap` do? Give an example.**

> `vmap` (vectorized map) automatically batches a function written for a single sample to work on a batch without Python loops. Example:
> ```python
> def process_one(x):  # works on shape (D,)
>     return jnp.dot(x, x)  # scalar
> 
> process_batch = vmap(process_one)  # now works on (N, D) → (N,)
> X = jnp.ones((100, 20))  # 100 samples
> result = process_batch(X)  # shape (100,) — no Python loop!
> ```
> Unlike a Python loop, `vmap` generates a single vectorized XLA kernel — much faster and compiles cleanly with `jit`.

---

**Q5: JAX vs PyTorch — when would you choose JAX?**

> **Choose JAX when:**
> - You need maximum speed and are comfortable with functional programming
> - You need TPU support (JAX has first-class TPU support)
> - You're doing research requiring higher-order derivatives (`grad(grad(f))`) or custom gradient rules
> - You're working with probabilistic programming (NumPyro, BlackJAX)
> - You want to do physics simulations, ODE solving with backprop
> 
> **Choose PyTorch when:**
> - You want a rich ecosystem with mature libraries (HuggingFace, torchvision)
> - Your team is already familiar with PyTorch
> - You prefer stateful/mutable programming style
> - You need an easier debugging experience

---

**Q6: What is the "pure function" requirement in JAX?**

> JAX transformations (`jit`, `grad`, `vmap`) require **pure functions**: functions whose output depends only on their inputs, with no side effects (no global state mutation, no I/O). This is because:
> - `jit` traces the function once and caches the compiled graph — side effects only run during tracing
> - `grad` needs to run the function forward and backward — side effects would be applied twice
> - `vmap` maps the function over a batch — side effects would be applied per-element
> 
> Practical implication: use `jax.debug.print()` instead of Python `print()` inside JIT; use `lax.cond` instead of Python `if` on JAX values; use `lax.scan` instead of Python `for` loops over JAX values.

## 14. Resources

### Official
- **JAX Documentation:** https://docs.jax.dev/
- **JAX GitHub:** https://github.com/google/jax
- **JAX Tutorial Notebooks:** https://jax.readthedocs.io/en/latest/tutorials.html

### Ecosystem
- **Flax (NN library):** https://flax.readthedocs.io/
- **Optax (optimizers):** https://optax.readthedocs.io/
- **Equinox (PyTorch-style on JAX):** https://docs.kidger.site/equinox/
- **NumPyro (probabilistic programming):** https://num.pyro.ai/
- **BlackJAX (Bayesian inference):** https://blackjax-devs.github.io/blackjax/

### Videos
- **JAX: What's it all about? (Google I/O):** https://www.youtube.com/watch?v=iDxJxIyzSiM
- **JAX tutorial by Aleksa Gordić:** https://www.youtube.com/watch?v=SstuvS-tVc0

### Papers
- **JAX paper:** https://arxiv.org/abs/2012.16867
- **Flax paper:** https://arxiv.org/abs/2402.19427

## 15. Summary & What's Next

### What You Learned

| Concept | Key Takeaway |
|---|---|
| **jnp arrays** | Like NumPy but immutable; use `.at[].set()` for updates |
| **jax.grad** | `grad(f)` → a function computing ∂f/∂x; compose with `jit` for fast training |
| **jax.jit** | Compiles a function to XLA machine code; first call traces, rest are fast |
| **jax.vmap** | Automatically vectorize a single-sample function over a batch |
| **PRNG keys** | Always `random.split(key)` before use — functional, reproducible |
| **Pure functions** | No side effects; use `lax.cond/scan` instead of Python `if/for` on JAX values |
| **Flax** | `nn.Module` with **explicit** parameter state (not stored in the model) |
| **Optax** | Composable optimizers + LR schedules + gradient clipping |

### The Deep Learning Framework Landscape

```
JAX            → Research frontier, maximum speed, TPUs, functional style
PyTorch        → Research standard, flexible, huge ecosystem, easier debugging
TensorFlow     → Production, mobile, browser deployment, mature MLOps tooling
Keras 3        → High-level API that runs on ANY of the above backends
```

### Phase 3 Complete!

You've now covered all of **Classical Machine Learning** and **Deep Learning**:
- Scikit-Learn, XGBoost, LightGBM, CatBoost — tree-based ML
- PyTorch, TensorFlow, Keras, JAX — deep learning

**Up next — Phase 4:**
- **NLP**: spaCy, NLTK, HuggingFace Transformers, Sentence-Transformers
- **Computer Vision**: OpenCV, Torchvision, YOLO, image augmentation